# MOFA-Flex-GraphGP evaluation

### What's different from plain MOFA-Flex

The `evaluate_mofaflex_multiomic_clinical.ipynb`, which this notebook depends on for Part 6-style comparisons downstream, in `evaluate_imputation_methods_clinical.ipynb`: the loadings prior $W$ is no longer a generic spike-and-slab sparsity prior - it's `mfl.priors.GraphGP`, built directly from a
STRING protein-protein-interaction (PPI) network's graph Laplacian eigenbasis (`mfl.tl.graph_feature_basis`). Every other modeling choice (`factor_prior="Normal"`, SVI/Pyro inference, `n_particles=5`, `early_stopper_patience=200`) is identical to plain MOFA-Flex - this notebook isolates the effect of the graph-informed prior alone.

### Autodetection of feature-ID mapping.

STRING's own tools (`string_interactions`, `features_to_adjacency`) operate on gene symbols. This project's `dlbcl_protein` and `crc_protein` matrices are indexed by UniProt accessions (verified
against `evaluate_mofaplus_multiomic_clinical.ipynb` / `benchmark_imputation_clinical.ipynb`'s
own `Protein_ID` convention), so those two get the same UniProt -> STRING-ID -> gene-symbol. For the `dlbcl_rna`'s feature-ID format **Part 1 below (Cell 5) detects
the format at runtime** (Ensembl gene ID / UniProt accession / already-a-gene-symbol, via
regex against the actual `var_names`) and picks the matching mapping path
automatically, printing what it detected and how many features it could map. If the
auto-detected path doesn't yield good STRING coverage for `dlbcl_rna` (see the
`graph_coverage` diagnostic Cell 5 prints), that's the signal to hardcode the format
instead of relying on the detector - the detection logic and its regexes are isolated in
one function (`detect_id_format`) specifically so that's a one-line override.

### All three matrices get MOFA-Flex-GraphGP 
so it's directly comparable to MOFA+ and MOFA-Flex everywhere they're both scored.

### Requirements

everything `evaluate_mofaflex_multiomic_clinical.ipynb` requires
(`mofaflex` + CUDA `torch`), **plus** the GraphGP fork:
`pip install git+https://github.com/MLO-lab/mofaflex-graphgp.git`, **plus** two STRING
flat files locally: `9606.protein.aliases.v12.0.txt.gz` and `9606.protein.info.v12.0.txt.gz`
(https://string-db.org/cgi/download - human, v12.0.  `protein.links` is NOT needed, since
`string_interactions` fetches edges live). Cell 1 looks for these under a `string_db`
directory resolved the same candidate-path way as `PROCESSED_DIR`/`RESULTS_DIR`; if none of
the candidates has the two files, every graph-dependent cell below skips cleanly with a
clear message rather than failing, exactly like the `MOFAFLEX_AVAILABLE` guard already
does for missing `mofaflex`/`torch`.

### Checkpointing 

writes into the SAME shared `pilot_checkpoint.csv` /
`benchmark_checkpoint_<matrix>.csv` files as the 25 conventional methods, MOFA+, and
MOFA-Flex - `MOFA-Flex-GraphGP` shows up as one more `Method` in the same tables, not a
separate analysis to cross-reference.

In [1]:
# --- Cell 1: cap CPU thread usage, imports, path resolution, mofaflex/GraphGP guard ---
import os
import time
import gzip
import logging
import warnings
from pathlib import Path
from collections.abc import Mapping


def _cluster_cpu_budget(default=4):
    """Same priority order as the rest of this project's notebooks: an explicit
    IMPUTATION_CPU_BUDGET override beats the Slurm allocation, which beats this
    process's own CPU affinity, which beats a conservative default."""
    override = os.environ.get("IMPUTATION_CPU_BUDGET")
    if override:
        try:
            return max(1, int(override)), "IMPUTATION_CPU_BUDGET override"
        except ValueError:
            print(f"IMPUTATION_CPU_BUDGET={override!r} isn't a valid integer - ignoring it.")

    for var in ("SLURM_CPUS_PER_TASK", "SLURM_CPUS_ON_NODE"):
        val = os.environ.get(var)
        if val:
            try:
                return max(1, int(val)), var
            except ValueError:
                pass
    val = os.environ.get("SLURM_JOB_CPUS_PER_NODE")
    if val:
        import re
        m = re.match(r"(\d+)", val)
        if m:
            return max(1, int(m.group(1))), "SLURM_JOB_CPUS_PER_NODE"

    try:
        return max(1, len(os.sched_getaffinity(0))), "CPU affinity (no Slurm allocation, no override detected)"
    except AttributeError:
        return max(1, os.cpu_count() or default), "os.cpu_count() (no Slurm allocation, no override detected)"


N_JOBS, _cpu_budget_source = _cluster_cpu_budget()
for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = str(N_JOBS)
print(f"CPU budget for this session: {N_JOBS} threads (source: {_cpu_budget_source})")

import numpy as np
import pandas as pd
import mudata as mu

mu.set_options(pull_on_update=False)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("mofaflex_graphgp")

RNG_SEED = 0  # keep in sync with benchmark_imputation_clinical.ipynb/py /
              # evaluate_mofaplus_multiomic_clinical.ipynb / evaluate_mofaflex_multiomic_clinical.ipynb

# --- mofaflex + GraphGP guard -- same MOFAFLEX_AVAILABLE pattern as
# evaluate_mofaflex_multiomic_clinical.ipynb's Cell 1.
RUN_MOFAFLEX = True
MOFAFLEX_AVAILABLE = False
GRAPHGP_AVAILABLE = False
DEVICE = "cpu"
if RUN_MOFAFLEX:
    try:
        import torch
        import mofaflex as mfl
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        try:
            _mfl_ver = mfl.__version__
        except AttributeError:
            _mfl_ver = "unknown"

        GRAPHGP_AVAILABLE = hasattr(mfl.priors, "GraphGP") and all(
            hasattr(mfl.tl, fn) for fn in
            ("string_interactions", "features_to_adjacency", "graph_coverage", "graph_feature_basis")
        )
        MOFAFLEX_AVAILABLE = GRAPHGP_AVAILABLE  # no fallback path without GraphGP in this notebook

        print(f"mofaflex {_mfl_ver} available | torch {torch.__version__} | device: {DEVICE}")
        if GRAPHGP_AVAILABLE:
            print("GraphGP prior and mfl.tl graph utilities: present.")
        else:
            print("GraphGP prior and/or mfl.tl graph utilities: MISSING. Install with:\n"
                  "  pip install git+https://github.com/MLO-lab/mofaflex-graphgp.git\n"
                  "The graph-dependent cells below are guarded and will skip cleanly until then.")
        if DEVICE == "cpu":
            print("  no CUDA device detected - SVI on CPU is much slower than on GPU. "
                  "Time the pilot cell before committing to the full grid.")
    except Exception as exc:
        print(f"MOFA-Flex-GraphGP disabled ({type(exc).__name__}: {exc}). Requires `mofaflex` "
              f"(Python >= 3.11) + a matching CUDA `torch` build + the GraphGP fork "
              f"(pip install git+https://github.com/MLO-lab/mofaflex-graphgp.git). The rest of "
              f"this notebook (matrix rebuild, masking) will still run, but every training cell "
              f"below will skip with a clear message.")

try:
    THIS_DIR = Path(__file__).resolve().parent
except NameError:
    THIS_DIR = Path.cwd()

BASE_DIR = THIS_DIR.parent  # the project root, e.g. /home/rnjue/Imputationv2

CANDIDATE_PROCESSED_DIRS = []
if os.environ.get("IMPUTATION_OUTPUT_DIR"):
    CANDIDATE_PROCESSED_DIRS.append(Path(os.environ["IMPUTATION_OUTPUT_DIR"]).expanduser())
CANDIDATE_PROCESSED_DIRS += [
    BASE_DIR / "data" / "processed",
    Path.home() / "reuben_imputation" / "processed",
    Path("/projects/datasets_BIO/reuben_imputation/processed"),
    THIS_DIR / "processed",
]
PROCESSED_DIR = next((d for d in CANDIDATE_PROCESSED_DIRS if d.exists()), None)
if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "no 'processed' directory found among: " + ", ".join(str(d) for d in CANDIDATE_PROCESSED_DIRS) +
        ". Run read_multiomic_Data_clinical.ipynb through Cell 14 first, or set IMPUTATION_OUTPUT_DIR."
    )

CANDIDATE_RESULTS_DIRS = [
    BASE_DIR / "results",
    PROCESSED_DIR.parent / "imputation_results",
    Path.home() / "reuben_imputation" / "imputation_results",
]
RESULTS_DIR = next((d for d in CANDIDATE_RESULTS_DIRS if d.exists()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError(
        "no imputation_results directory found among: " + ", ".join(str(d) for d in CANDIDATE_RESULTS_DIRS) +
        ". Run benchmark_imputation_clinical.ipynb first."
    )
# Same shared files evaluate_mofaplus_multiomic_clinical.ipynb / evaluate_mofaflex_multiomic_clinical.ipynb
# write to - MOFA-Flex-GraphGP's rows get appended alongside theirs, all in one table.
PILOT_CHECKPOINT_PATH = RESULTS_DIR / "pilot_checkpoint.csv"
PILOT_MATRICES_DIR = RESULTS_DIR / "pilot_imputed_matrices"
PILOT_MATRICES_DIR.mkdir(exist_ok=True)

# STRING flat-file directory - same candidate-path resolution style as PROCESSED_DIR/
# RESULTS_DIR above. Point IMPUTATION_STRING_DIR at wherever you downloaded
# 9606.protein.aliases.v12.0.txt.gz and 9606.protein.info.v12.0.txt.gz if none of these
# default locations fit.
CANDIDATE_STRING_DIRS = []
if os.environ.get("IMPUTATION_STRING_DIR"):
    CANDIDATE_STRING_DIRS.append(Path(os.environ["IMPUTATION_STRING_DIR"]).expanduser())
CANDIDATE_STRING_DIRS += [
    BASE_DIR / "data" / "string_db",
    PROCESSED_DIR / "string_db",
    Path.home() / "reuben_imputation" / "string_db",
    Path("/projects/datasets_BIO/reuben_imputation/string_db"),
    THIS_DIR / "string_db",
]
STRING_DIR = next((d for d in CANDIDATE_STRING_DIRS if d.exists()), CANDIDATE_STRING_DIRS[0])
STRING_DIR.mkdir(parents=True, exist_ok=True)

print("processed dir:", PROCESSED_DIR, "| results dir:", RESULTS_DIR, "| string dir:", STRING_DIR)

mdata = mu.read_h5mu(PROCESSED_DIR / "dlbcl_mudata_filtered.h5mu")
crc_df = pd.read_csv(PROCESSED_DIR / "crc_proteomics_samples_x_features.csv", index_col=0)


def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)


CPU budget for this session: 80 threads (source: CPU affinity (no Slurm allocation, no override detected))


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

mofaflex 0.1.dev1+gdf5a17c14 available | torch 2.11.0+cu130 | device: cpu
GraphGP prior and mfl.tl graph utilities: present.
  no CUDA device detected - SVI on CPU is much slower than on GPU. Time the pilot cell before committing to the full grid.
processed dir: /home/rnjue/Imputationv2/data/processed | results dir: /home/rnjue/Imputationv2/results | string dir: /home/rnjue/Imputationv2/data/string_db


In [2]:
# --- Cell 2: rebuild the three per-matrix truth matrices (identical to
# evaluate_mofaflex_multiomic_clinical.ipynb's Cell 2 / evaluate_mofaplus_multiomic_
# clinical.ipynb's Cell 2 / benchmark_imputation_clinical.ipynb's Cell 3 - same
# 500-most-variable-feature subset, so MOFA-Flex-GraphGP is scored on exactly the
# same matrices as every other method in this project).
rna_adata = mdata.mod["1_rna"]
assert (rna_adata.obs_names == mdata.obs_names).all()
profiled_mask = mdata.obs["rna_profiled"].to_numpy()

rna_full = pd.DataFrame(
    to_dense(rna_adata.X)[profiled_mask].T, index=rna_adata.var_names, columns=rna_adata.obs_names[profiled_mask]
)
prot_full = pd.DataFrame(to_dense(mdata.mod["2_prot"].X).T, index=mdata.mod["2_prot"].var_names, columns=mdata.obs_names)
crc_full = crc_df.T

N_TOP_FEATURES = 500  # matches every other notebook/script in this project


def top_variable_features(df, n):
    if df.shape[0] <= n:
        return df
    top_idx = df.var(axis=1, skipna=True).sort_values(ascending=False).index[:n]
    return df.loc[df.index.isin(top_idx)]


MATRICES = {
    "dlbcl_rna": top_variable_features(rna_full, N_TOP_FEATURES),
    "dlbcl_protein": top_variable_features(prot_full, N_TOP_FEATURES),
    "crc_protein": top_variable_features(crc_full, N_TOP_FEATURES),
}
# ID convention per matrix - drives which mapping path Cell 5 uses. dlbcl_protein and
# crc_protein are confirmed UniProt accessions (same Protein_ID convention as
# evaluate_mofaplus_multiomic_clinical.ipynb / benchmark_imputation_clinical.ipynb).
# dlbcl_rna is left as "auto" - Cell 5 detects it from the actual var_names at runtime
# (see this notebook's header markdown for why).
MATRIX_ID_TYPE = {"dlbcl_rna": "auto", "dlbcl_protein": "uniprot", "crc_protein": "uniprot"}
for name, df in MATRICES.items():
    print(f"{name}: {df.shape}  (ID type: {MATRIX_ID_TYPE[name]})")


dlbcl_rna: (500, 318)  (ID type: auto)
dlbcl_protein: (500, 332)  (ID type: uniprot)
crc_protein: (500, 997)  (ID type: uniprot)


In [3]:
# --- Cell 3: masking functions and PILOT_ROW (the same as from
# evaluate_mofaflex_multiomic_clinical.ipynb's Cell 3 - same masking design, so
# MOFA-Flex-GraphGP's scenarios are byte-identical to MOFA-Flex's, MOFA+'s, and the
# 25 methods').
def generate_mnar_weighted_positions(X, n_to_mask, slope=-1.5, rng=None, exclude_mask=None):
    if rng is None:
        rng = np.random.default_rng()
    eligible = X.notna().values
    if exclude_mask is not None:
        eligible = eligible & ~exclude_mask.values
    positions = np.argwhere(eligible)
    if n_to_mask == 0:
        return np.empty((0, 2), dtype=int)
    values = X.values[eligible]
    mean_v, std_v = values.mean(), values.std()
    z = np.zeros_like(values) if std_v == 0 else (values - mean_v) / std_v
    weights = np.exp(slope * z)
    weights = weights / weights.sum()
    selected = rng.choice(len(positions), size=n_to_mask, replace=False, p=weights)
    return positions[selected]


def generate_mixed_mask(X, masking_fraction=0.20, mnar_fraction=0.50, slope=-1.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    eligible = X.notna().values
    n_eligible = eligible.sum()
    total_to_mask = int(round(masking_fraction * n_eligible))
    n_mnar = int(round(total_to_mask * mnar_fraction))
    n_mcar = total_to_mask - n_mnar

    mnar_positions = generate_mnar_weighted_positions(X, n_mnar, slope=slope, rng=rng)
    mnar_excl = np.zeros(X.shape, dtype=bool)
    if len(mnar_positions) > 0:
        mnar_excl[mnar_positions[:, 0], mnar_positions[:, 1]] = True
    mnar_excl_df = pd.DataFrame(mnar_excl, index=X.index, columns=X.columns)

    mcar_positions = np.empty((0, 2), dtype=int)
    if n_mcar > 0:
        available = np.argwhere(eligible & ~mnar_excl_df.values)
        n_mcar_clamped = min(n_mcar, len(available))
        mcar_positions = available[rng.choice(len(available), size=n_mcar_clamped, replace=False)]

    mask = np.zeros(X.shape, dtype=bool)
    mechanism = np.full(X.shape, "", dtype=object)
    if len(mnar_positions) > 0:
        mask[mnar_positions[:, 0], mnar_positions[:, 1]] = True
        mechanism[mnar_positions[:, 0], mnar_positions[:, 1]] = "MNAR"
    if len(mcar_positions) > 0:
        mask[mcar_positions[:, 0], mcar_positions[:, 1]] = True
        mechanism[mcar_positions[:, 0], mcar_positions[:, 1]] = "MCAR"
    return (
        pd.DataFrame(mask, index=X.index, columns=X.columns),
        pd.DataFrame(mechanism, index=X.index, columns=X.columns),
    )


def build_masked_scenario(row, X_truth, slope=-1.5):
    rng = np.random.default_rng(int(row["Seed"]))
    mask, mechanism = generate_mixed_mask(
        X_truth, masking_fraction=float(row["Masking_Fraction"]), mnar_fraction=float(row["MNAR_Fraction"]),
        slope=slope, rng=rng,
    )
    return X_truth.mask(mask), mask, mechanism


PILOT_ROW = pd.Series({"Masking_Fraction": 0.10, "MNAR_Fraction": 0.0, "Replicate": 1, "Seed": RNG_SEED})


In [4]:
# --- Cell 4: metrics --- calculate_metrics, pooled_correlations, score_full
# (the same as from evaluate_mofaflex_multiomic_clinical.ipynb's Cell 4).
from scipy.stats import spearmanr


def calculate_metrics(truth, prediction, mask):
    y_true_mat, y_pred_mat, mask_mat = truth.values, prediction.values, mask.values
    y_true, y_pred = y_true_mat[mask_mat], y_pred_mat[mask_mat]
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]

    errors = y_true - y_pred
    rmse = np.sqrt(np.mean(errors ** 2)) if len(errors) else np.nan
    mae = np.mean(np.abs(errors)) if len(errors) else np.nan
    pearson_r = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan

    nrmse_vals = []
    min_masked_per_feature = 5
    for i in range(y_true_mat.shape[0]):
        row_mask = mask_mat[i]
        if row_mask.sum() < min_masked_per_feature:
            continue
        yt, yp = y_true_mat[i, row_mask], y_pred_mat[i, row_mask]
        v = np.isfinite(yt) & np.isfinite(yp)
        yt, yp = yt[v], yp[v]
        if len(yt) < min_masked_per_feature:
            continue
        feat_range = np.nanmax(y_true_mat[i]) - np.nanmin(y_true_mat[i])
        if not np.isfinite(feat_range) or feat_range <= 0:
            continue
        row_rmse = np.sqrt(np.mean((yt - yp) ** 2))
        nrmse_vals.append(row_rmse / feat_range)

    return {
        "RMSE": rmse, "MAE": mae, "Pearson_r": pearson_r,
        "NRMSE": float(np.mean(nrmse_vals)) if nrmse_vals else np.nan,
        "N_Evaluated": len(y_true),
    }


def pooled_correlations(X_truth, X_hat, mask, min_masked_per_feature=5):
    yt_mat, yp_mat, mask_mat = X_truth.values, X_hat.values, mask.values
    raw_true, raw_pred, cen_true, cen_pred = [], [], [], []
    n_features_used = 0
    for i in range(yt_mat.shape[0]):
        row_mask = mask_mat[i]
        if row_mask.sum() < min_masked_per_feature:
            continue
        yt, yp = yt_mat[i, row_mask], yp_mat[i, row_mask]
        v = np.isfinite(yt) & np.isfinite(yp)
        yt, yp = yt[v], yp[v]
        if len(yt) < min_masked_per_feature:
            continue
        raw_true.append(yt)
        raw_pred.append(yp)
        cen_true.append(yt - yt.mean())
        cen_pred.append(yp - yp.mean())
        n_features_used += 1

    if n_features_used == 0:
        return {"Pearson_r_raw_pooled": np.nan, "Spearman_r_raw_pooled": np.nan,
                "Pearson_r_feature_centered_pooled": np.nan, "Spearman_r_feature_centered_pooled": np.nan,
                "N_Features_Pooled": 0}

    raw_true, raw_pred = np.concatenate(raw_true), np.concatenate(raw_pred)
    cen_true, cen_pred = np.concatenate(cen_true), np.concatenate(cen_pred)

    def _pearson(a, b):
        return np.corrcoef(a, b)[0, 1] if len(a) > 1 and np.std(a) > 0 and np.std(b) > 0 else np.nan

    def _spearman(a, b):
        return spearmanr(a, b).correlation if len(a) > 1 and np.std(a) > 0 and np.std(b) > 0 else np.nan

    return {
        "Pearson_r_raw_pooled": _pearson(raw_true, raw_pred),
        "Spearman_r_raw_pooled": _spearman(raw_true, raw_pred),
        "Pearson_r_feature_centered_pooled": _pearson(cen_true, cen_pred),
        "Spearman_r_feature_centered_pooled": _spearman(cen_true, cen_pred),
        "N_Features_Pooled": n_features_used,
    }


def score_full(matrix_name, method_name, X_truth, X_hat, mask, t0, scenario_row=None):
    """Bundles calculate_metrics + pooled_correlations into one row, in the
    same shape pilot_checkpoint.csv expects. scenario_row defaults to
    PILOT_ROW (pilot usage); full-grid usage passes the scenario's own row."""
    row = PILOT_ROW if scenario_row is None else scenario_row
    X_hat = X_hat.reindex(index=X_truth.index, columns=X_truth.columns)
    m = calculate_metrics(X_truth, X_hat, mask)
    m.update(pooled_correlations(X_truth, X_hat, mask))
    m.update({"Matrix": matrix_name, "Method": method_name, "Seconds": time.time() - t0,
              **{c: row[c] for c in ["Masking_Fraction", "MNAR_Fraction", "Replicate", "Seed"]}})
    return m, X_hat


### Part 1 - Build the STRING graph eigenbasis, per matrix

It loops over all three matrices instead of one proteomics matrix, and also it auto-detects each matrix's feature-ID format instead of assuming UniProt (see this notebook's header markdown).

STRING's `string_interactions`/`features_to_adjacency` operate on gene symbols, so
whatever the source ID format, it goes through the same three-stage pipeline: detect the
ID format -> map to a STRING internal ID -> map that to a gene symbol (`protein.aliases`
then `protein.info`). Symbols get built into a PPI graph and its Laplacian eigenbasis
(`mfl.tl.graph_feature_basis`); features that don't resolve to a symbol (or whose symbol
isn't in the graph) fall back to the basis centroid.

In [5]:
# --- Cell 5: STRING PPI graph eigenbasis, built per matrix 
import re
import os

N_GRAPH_BASIS = 150     
SCORE_THRESHOLD = 700    # STRING high-confidence cutoff; also string_interactions' own default
SPECIES = 9606            # human


def _open_maybe_gz(path):
    return gzip.open(path, "rt") if path.suffix == ".gz" else open(path, "r")


def _strip_common_prefix(ids):
    """Detects and strips a shared leading '<token>_' prefix baked into every ID. e.g.
    dlbcl_rna's real var_names embed the source MuData modality key literally in the
    string ('1_rna_CPNE5', '1_rna_TP53', ...), with the actual gene symbol after the last
    such prefix. Uses the longest literal common prefix across the whole id list, trimmed
    back to the last underscore inside it so a token is never cut mid-word. Returns
    (stripped_ids, prefix_or_None) - a safe no-op (prefix=None, ids unchanged) for IDs
    that don't share one, e.g. this project's UniProt-accession protein matrices."""
    ids = [str(x) for x in ids]
    if not ids:
        return ids, None
    common = os.path.commonprefix(ids)
    last_us = common.rfind("_")
    if last_us < 1:
        return ids, None
    prefix = common[:last_us + 1]
    return [s[len(prefix):] for s in ids], prefix


# --- ID-format auto-detection -------------------------------------------------
# Isolated in one function on purpose: if the auto-detected path gives dlbcl_rna poor
# STRING coverage (see the graph_coverage diagnostic printed below), override by setting
# MATRIX_ID_TYPE["dlbcl_rna"] directly (Cell 2) to "ensembl", "uniprot", or "gene_symbol"
# and re-running this cell - no need to touch the detection regexes themselves.
_ENSEMBL_RE = re.compile(r"^ENSG\d{11}(\.\d+)?$")
_UNIPROT_RE = re.compile(
    r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})(-\d+)?$"
)
_GENE_SYMBOL_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9.\-]{0,14}$")


def detect_id_format(feature_ids, sample_size=200):
    """Regex-classifies a sample of feature_ids against Ensembl-gene, UniProt-accession,
    and (loosely) gene-symbol patterns; returns whichever pattern the largest share of the
    sample matches, defaulting to "gene_symbol" when nothing matches confidently (the
    common case for RNA var_names, and a safe no-op for build_id_to_symbol_map: an
    already-a-symbol feature just maps to itself). Strips a shared ID prefix first (see
    _strip_common_prefix) so a modality-prefixed symbol like "1_rna_CPNE5" is classified
    on "CPNE5", not the full string. Prints the vote counts so the choice is auditable,
    not a silent guess."""
    ids = [str(x).split(";")[0].strip() for x in feature_ids]
    stripped_ids, prefix = _strip_common_prefix(ids)
    if prefix:
        print(f"detect_id_format: stripped shared prefix '{prefix}' from every ID before "
              f"classifying (e.g. '{ids[0]}' -> '{stripped_ids[0]}').")
    sample = (stripped_ids if len(stripped_ids) <= sample_size
              else list(pd.Series(stripped_ids).sample(sample_size, random_state=0)))
    n = len(sample)
    votes = {
        "ensembl": sum(bool(_ENSEMBL_RE.match(s)) for s in sample),
        "uniprot": sum(bool(_UNIPROT_RE.match(s)) for s in sample),
        "gene_symbol": sum(bool(_GENE_SYMBOL_RE.match(s)) and not _ENSEMBL_RE.match(s)
                            and not _UNIPROT_RE.match(s) for s in sample),
    }
    best = max(votes, key=votes.get)
    if votes[best] == 0:
        # True zero-vote tie - max() would otherwise silently return "ensembl" (the
        # first key in the dict) rather than this function's own documented default.
        best = "gene_symbol"
    frac = votes[best] / n if n else 0.0
    print(f"detect_id_format: sampled {n} feature IDs - "
          f"ensembl={votes['ensembl']}, uniprot={votes['uniprot']}, gene_symbol={votes['gene_symbol']} "
          f"-> detected '{best}' ({frac:.0%} of sample). Example IDs: {sample[:5]}")
    if frac < 0.5:
        print(f"  low-confidence detection ({frac:.0%}) - if graph_coverage below looks poor, "
              f"set MATRIX_ID_TYPE for this matrix explicitly (Cell 2) and re-run this cell.")
    return best


def map_id_to_string_id(aliases_path, ids):
    """Parse STRING's protein.aliases file: columns `#string_protein_id  alias  source`
    (tab-separated). Works for ANY alias type present in that file (UniProt, Ensembl,
    RefSeq, ...) since it only matches the requested `ids` against the alias column -
    not UniProt-specific despite the name of the file it reads."""
    mapping = {}
    n_lines = 0
    with _open_maybe_gz(aliases_path) as fh:
        fh.readline()  # header
        for line in fh:
            n_lines += 1
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            string_id, alias = parts[0], parts[1]
            if alias in ids and alias not in mapping:
                mapping[alias] = string_id
    log.info("Scanned %d alias rows; mapped %d/%d requested IDs to a STRING ID.",
              n_lines, len(mapping), len(ids))
    return mapping


def map_string_id_to_symbol(info_path):
    """Parse STRING's protein.info file: columns `#string_protein_id  preferred_name
    protein_size  annotation` (tab-separated). `preferred_name` is the gene symbol -
    the same field string_interactions itself resolves symbols from."""
    mapping = {}
    with _open_maybe_gz(info_path) as fh:
        fh.readline()  # header
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            mapping[parts[0]] = parts[1]
    log.info("Parsed %d STRING-ID -> gene-symbol entries from protein.info.", len(mapping))
    return mapping


def build_id_to_symbol_map(cohort, id_type, string_dir):
    """cohort entries may be ';'-joined ID groups (same convention as the UniProt
    Protein_ID index elsewhere in this project). Returns (id_group -> gene symbol dict,
    unmapped list), keyed by the ORIGINAL (unstripped) cohort entries throughout, even
    though a shared prefix (see _strip_common_prefix) is stripped internally before
    mapping. id_type="gene_symbol" is a trivial identity map on the stripped ID - no
    STRING alias lookup needed, since the stripped ID already IS the thing
    string_interactions wants."""
    cohort_list = [str(x) for x in cohort]
    stripped_cohort, prefix = _strip_common_prefix(cohort_list)
    if prefix:
        log.info("build_id_to_symbol_map: stripped shared prefix '%s' from every cohort ID "
                  "before mapping (e.g. '%s' -> '%s').", prefix, cohort_list[0], stripped_cohort[0])

    group_of = {}
    for original_fid, stripped_fid in zip(cohort, stripped_cohort):
        group_of[original_fid] = str(stripped_fid).split(";")

    if id_type == "gene_symbol":
        pid_to_symbol = {fid: parts[0] for fid, parts in group_of.items()}
        return pid_to_symbol, []

    all_ids = set()
    for parts in group_of.values():
        all_ids.update(parts)

    aliases_path = string_dir / "9606.protein.aliases.v12.0.txt.gz"
    if not aliases_path.exists():
        aliases_path = string_dir / "9606.protein.aliases.v12.0.txt"
    info_path = string_dir / "9606.protein.info.v12.0.txt.gz"
    if not info_path.exists():
        info_path = string_dir / "9606.protein.info.v12.0.txt"

    id_to_string = map_id_to_string_id(aliases_path, all_ids)
    string_to_symbol = map_string_id_to_symbol(info_path)

    pid_to_symbol, unmapped = {}, []
    for fid, parts in group_of.items():
        sid = next((id_to_string[p] for p in parts if p in id_to_string), None)
        symbol = string_to_symbol.get(sid) if sid is not None else None
        if symbol is not None:
            pid_to_symbol[fid] = symbol
        else:
            unmapped.append(fid)
    log.info("Resolved %d/%d features to a gene symbol (%d unmapped -> centroid fallback).",
              len(pid_to_symbol), len(cohort), len(unmapped))
    return pid_to_symbol, unmapped


def build_graph_eigenbasis_for_matrix(matrix_name, cohort, id_type, string_dir,
                                       eigvecs_out_path, eigvals_out_path,
                                       n_components=N_GRAPH_BASIS, species=SPECIES,
                                       score_threshold=SCORE_THRESHOLD):
    if id_type == "auto":
        id_type = detect_id_format(cohort)
        print(f"{matrix_name}: auto-detected ID type = '{id_type}'")

    pid_to_symbol, unmapped = build_id_to_symbol_map(cohort, id_type, string_dir)

    # Multiple feature IDs can (rarely) resolve to the same gene symbol - dedupe for the
    # graph-building calls, then re-expand back to full cohort order below.
    unique_symbols = sorted(set(pid_to_symbol.values()))
    n_collapsed = len(pid_to_symbol) - len(unique_symbols)
    if n_collapsed > 0:
        log.info("%s: %d features share a gene symbol with another feature in the cohort "
                  "(%d unique symbols from %d mapped features) - they'll share an eigenvector row.",
                  matrix_name, n_collapsed, len(unique_symbols), len(pid_to_symbol))

    edges = mfl.tl.string_interactions(features=unique_symbols, species=species, score_threshold=score_threshold)
    adjacency = mfl.tl.features_to_adjacency(unique_symbols, edges)
    coverage = mfl.tl.graph_coverage(unique_symbols, adjacency)
    print(f"{matrix_name}: graph coverage diagnostics (mfl.tl.graph_coverage):")
    for k, v in coverage.items():
        print(f"  {k}: {v}")
    if isinstance(coverage, dict) and coverage.get("worth_trying") is False:
        log.warning("%s: graph_coverage reports worth_trying=False - the PPI graph may be too "
                    "sparse for this cohort/id_type/score_threshold to give GraphGP anything "
                    "useful to work with. If id_type was auto-detected, this is the signal to "
                    "override MATRIX_ID_TYPE (Cell 2) instead of trusting the detector.", matrix_name)

    # mfl.tl.graph_feature_basis requires 0 < n_components < n_features - guard this
    # explicitly (with a clear, actionable message) instead of letting it raise a bare
    # ValueError deep inside mofaflex.
    if len(unique_symbols) <= n_components:
        log.warning("%s: only %d feature(s) resolved to a gene symbol - too few to build a "
                    "%d-component graph eigenbasis (mfl.tl.graph_feature_basis requires "
                    "0 < n_components < n_features). Skipping this matrix's GraphGP prior "
                    "rather than crashing on that call. If id_type was auto-detected, this "
                    "usually means the wrong id_type was picked - check the ID-format "
                    "diagnostics printed above, override MATRIX_ID_TYPE (Cell 2) if needed, "
                    "and re-run this cell.", matrix_name, len(unique_symbols), n_components)
        return None, None

    eigvecs_arr, eigvals = mfl.tl.graph_feature_basis(adjacency, n_components=n_components)
    symbol_row = {s: i for i, s in enumerate(unique_symbols)}
    centroid = eigvecs_arr.mean(axis=0)

    rows = []
    for fid in cohort:
        symbol = pid_to_symbol.get(fid)
        rows.append(eigvecs_arr[symbol_row[symbol]] if symbol in symbol_row else centroid)
    eigvecs_df = pd.DataFrame(np.stack(rows), index=cohort,
                               columns=[f"eig_{i}" for i in range(eigvecs_arr.shape[1])])
    eigvecs_df.index.name = "Feature_ID"

    eigvecs_df.to_csv(eigvecs_out_path)
    np.save(eigvals_out_path, np.asarray(eigvals))
    log.info("%s: saved %s (%d features x %d eigenvectors) and %s. %d features used the centroid fallback.",
              matrix_name, eigvecs_out_path, *eigvecs_df.shape, eigvals_out_path, len(unmapped))
    return eigvecs_df, np.asarray(eigvals)


STRING_ALIAS_INFO_PRESENT = (
    (STRING_DIR / "9606.protein.aliases.v12.0.txt.gz").exists() or (STRING_DIR / "9606.protein.aliases.v12.0.txt").exists()
) and (
    (STRING_DIR / "9606.protein.info.v12.0.txt.gz").exists() or (STRING_DIR / "9606.protein.info.v12.0.txt").exists()
)

ppi_eigvecs_by_matrix, ppi_eigvals_by_matrix = {}, {}
if not (MOFAFLEX_AVAILABLE and GRAPHGP_AVAILABLE):
    print("Skipping graph construction: mofaflex-graphgp fork isn't installed (see Cell 1).")
elif not STRING_ALIAS_INFO_PRESENT:
    print("Skipping graph construction: 9606.protein.aliases.v12.0.txt.gz and/or "
          "9606.protein.info.v12.0.txt.gz not found under", STRING_DIR, "- download them from "
          "https://string-db.org/cgi/download (human, v12.0) and place them there, or point "
          "IMPUTATION_STRING_DIR at wherever you already have them. (protein.links is NOT "
          "needed - string_interactions fetches edges live.) The cells below are guarded and "
          "will skip cleanly until this is resolved.")
    for matrix_name in MATRICES:
        ppi_eigvecs_by_matrix[matrix_name], ppi_eigvals_by_matrix[matrix_name] = None, None
else:
    for matrix_name, X in MATRICES.items():
        eigvecs_path = RESULTS_DIR / f"graph_eigvecs_{matrix_name}.csv"
        eigvals_path = RESULTS_DIR / f"graph_eigvals_{matrix_name}.npy"
        if eigvecs_path.exists() and eigvals_path.exists():
            eigvecs_df = pd.read_csv(eigvecs_path, index_col="Feature_ID")
            eigvals = np.load(eigvals_path)
            print(f"{matrix_name}: loaded existing eigenbasis {eigvecs_df.shape} eigenvectors, "
                  f"{eigvals.shape} eigenvalues.")
        else:
            eigvecs_df, eigvals = build_graph_eigenbasis_for_matrix(
                matrix_name, list(X.index), MATRIX_ID_TYPE[matrix_name], STRING_DIR,
                eigvecs_path, eigvals_path,
            )
        ppi_eigvecs_by_matrix[matrix_name] = eigvecs_df
        ppi_eigvals_by_matrix[matrix_name] = eigvals


dlbcl_rna: loaded existing eigenbasis (500, 150) eigenvectors, (150,) eigenvalues.
dlbcl_protein: loaded existing eigenbasis (500, 150) eigenvectors, (150,) eigenvalues.
crc_protein: loaded existing eigenbasis (500, 150) eigenvectors, (150,) eigenvalues.


### Part 2 - The imputation function: `mofa_flex_graphgp_impute`

Uses `mfl.priors.GraphGP` as the weight prior, built directly from the graph eigenbasis
Part 1 computed for the matrix currently being scored. Everything else mirrors plain
MOFA-Flex's `mofa_flex_impute` (`evaluate_mofaflex_multiomic_clinical.ipynb`, Cell 5) -
same rescaling, same AnnData wrapping, same SVI settings. 

In [6]:
# --- Cell 6: mofa_flex_graphgp_impute() --- trains MOFA-Flex with the GraphGP weight
# prior and reconstructs the completed matrix. Generalized to take the eigenbasis as an argument 
# (so the SAME function works across dlbcl_rna/dlbcl_protein/crc_protein, each with its own eigenbasis from
# Part 1) instead of being hardcoded to one proteomics matrix + one global eigenbasis.
import anndata as ad
import contextlib, io as _io

N_FACTORS = 15  # matches evaluate_mofaflex_multiomic_clinical.ipynb / evaluate_mofaplus_multiomic_clinical.ipynb


def _normalize_lengthscales(x):
    """Normalize a lengthscale diagnostic into something print/round/np.save-friendly.
    GraphGP's `weight_graph_lengthscale` can come back as a torch tensor, a plain array, or
    a `mappingproxy`/dict keyed by data modality - recurse into any Mapping (mappingproxy
    included, since it satisfies the Mapping ABC) and detach/numpy-ify any tensor-like leaf,
    so callers only ever see a plain numpy array or a plain dict[str, numpy array]."""
    if isinstance(x, Mapping):
        return {k: _normalize_lengthscales(v) for k, v in x.items()}
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x)


def mofa_flex_graphgp_impute(Y_masked, ppi_eigvecs, ppi_eigvals, n_factors=N_FACTORS, seed=RNG_SEED,
                              max_epochs=2000, early_stopper_patience=200, n_particles=5,
                              kernel="diffusion", graph_confidence=0.9, device=None):
    """MOFA-Flex imputation with the real GraphGP prior on the WEIGHTS, built from the
    STRING PPI graph's Laplacian eigenbasis (Part 1, this matrix's own eigenbasis).

    Parameters
    ----------
    Y_masked : pandas.DataFrame
        features x samples, NaN = missing (real or masked). Works for dlbcl_rna,
        dlbcl_protein, or crc_protein identically, given that matrix's own eigenbasis.
    ppi_eigvecs : pandas.DataFrame
        index=Feature_ID, columns=eigenvector 0..n_components-1 (from Part 1, this matrix).
    ppi_eigvals : numpy.ndarray
        1D array of the matching Laplacian eigenvalues, same order as ppi_eigvecs' columns.

    Returns
    -------
    tuple[pandas.DataFrame, dict]
        Completed features x samples matrix, and a diagnostics dict (training loss trace,
        epochs run, per-factor R2 if available, graph-kernel lengthscale(s) if available).
    """
    if device is None:
        device = DEVICE

    Yc = Y_masked.T  # samples x features - mofaflex/AnnData convention

    # --- fully-missing-feature guard ---------------------------------------------
    # mofaflex's 'random' init computes
    # a per-feature statistic from that feature's observed entries, which is NaN for
    # a feature with zero observed entries and crashes the Normal likelihood's `loc`
    # parameter at initialization.Mirrors the fully-missing-row for
    # bpca_impute()/dreamai_impute() in benchmark_imputation_clinical.ipynb (Cell 12).
    fully_missing_features = Yc.isna().all(axis=0)
    if fully_missing_features.any():
        grand_mean = float(np.nanmean(Yc.values))
        if not np.isfinite(grand_mean):
            grand_mean = 0.0
        Yc = Yc.copy()
        Yc.loc[:, fully_missing_features] = grand_mean

    feature_shift = Yc.mean(axis=0, skipna=True)
    centered = (Yc - feature_shift).values
    global_scale = float(np.sqrt(np.nanvar(centered)))

    eigvecs_aligned = ppi_eigvecs.loc[Yc.columns].values.astype(np.float32)

    adata = ad.AnnData(X=Yc.values.astype(np.float32))
    adata.obs_names = Yc.index.astype(str)
    adata.var_names = Yc.columns.astype(str)
    adata.varm["graph_eigvecs"] = eigvecs_aligned
    data_dict = {"group1": {"features": adata}}

    weight_prior = mfl.priors.GraphGP(
        eigenvectors_varm_key="graph_eigvecs",
        eigenvalues=np.asarray(ppi_eigvals, dtype=np.float64),
        kernel=kernel,
        graph_confidence=graph_confidence,
    )

    with contextlib.redirect_stdout(_io.StringIO()):
        model = mfl.terms.MofaFlex(n_factors=n_factors, factor_prior="Normal", weight_prior=weight_prior)
        model.fit(
            data_dict, likelihoods="Normal", device=device, max_epochs=max_epochs,
            early_stopper_patience=early_stopper_patience, n_particles=n_particles,
            seed=seed, batch_size=0, plot_data_overview=False, subset_var=None,
            remove_constant_features=False, update_every=-1, save_path=False,
        )
        imputed = model.impute_data(data_dict, missing_only=False)

    Xhat_preprocessed = pd.DataFrame(
        np.asarray(imputed["group1"]["features"].X), index=Yc.index, columns=Yc.columns
    )
    Xhat = (Xhat_preprocessed * global_scale + feature_shift).T

    if fully_missing_features.any():
        # blank these features' predictions back to NaN - the model never saw a
        # real observation for them, so whatever it output is not a genuine
        # prediction and must not be scored as one (mirrors
        # _blank_out_unidentifiable_rows()'s role for bpca_impute()/dreamai_impute()).
        Xhat.loc[fully_missing_features[fully_missing_features].index, :] = np.nan

    diagnostics = {"model": model, "feature_shift": feature_shift, "global_scale": global_scale,
                    "n_fully_missing_features_blanked": int(fully_missing_features.sum())}
    diagnostics["training_loss"] = np.asarray(model.training_loss)
    diagnostics["n_epochs_run"] = int(len(diagnostics["training_loss"]))
    diagnostics["max_epochs"] = max_epochs
    diagnostics["early_stopped"] = diagnostics["n_epochs_run"] < max_epochs
    try:
        diagnostics["r2"] = model.get_r2()
    except Exception as exc:
        diagnostics["r2"] = None
        diagnostics["r2_error"] = f"{type(exc).__name__}: {exc}"

    try:
        diagnostics["weight_prior_lengthscales"] = _normalize_lengthscales(model.weight_graph_lengthscale)
    except Exception as exc:
        diagnostics["weight_prior_lengthscales"] = None
        diagnostics["lengthscale_error"] = f"{type(exc).__name__}: {exc}"
        diagnostics["_graph_like_attrs"] = sorted(
            a for a in dir(model) if ("graph" in a.lower() or "gp_" in a.lower()) and not a.startswith("_")
        )

    return Xhat, diagnostics


In [7]:
# --- Cell 7: pilot scenario, per-matrix baseline ("MOFA-Flex-GraphGP") -------------
# Mirrors evaluate_mofaflex_multiomic_clinical.ipynb's own pilot cell (Cell 6) -
# same PILOT_ROW, same shared pilot_checkpoint.csv / pilot_imputed_matrices/, so
# MOFA-Flex-GraphGP's pilot row lands right next to MOFA-Flex's, MOFA+'s, and the 25
# methods'. Skips a matrix cleanly if its eigenbasis wasn't built (Part 1).

if PILOT_CHECKPOINT_PATH.exists():
    _checkpoint_df = pd.read_csv(PILOT_CHECKPOINT_PATH)
    pilot_rows = _checkpoint_df.to_dict("records")
else:
    pilot_rows = []
_pilot_row_index = {(r["Matrix"], r["Method"]): i for i, r in enumerate(pilot_rows)}


def _pilot_matrix_path(matrix_name, method_name):
    return PILOT_MATRICES_DIR / f"pilot_{matrix_name}_imputed_{method_name}.csv"


def _record_pilot_result(matrix_name, method_name, m, X_hat):
    matrix_path = _pilot_matrix_path(matrix_name, method_name)
    X_hat.rename_axis("Feature_ID").to_csv(matrix_path)
    if (matrix_name, method_name) in _pilot_row_index:
        pilot_rows[_pilot_row_index[(matrix_name, method_name)]] = m
    else:
        pilot_rows.append(m)
        _pilot_row_index[(matrix_name, method_name)] = len(pilot_rows) - 1
    pd.DataFrame(pilot_rows).to_csv(PILOT_CHECKPOINT_PATH, index=False)


METHOD_NAME = "MOFA-Flex-GraphGP"

if not (MOFAFLEX_AVAILABLE and GRAPHGP_AVAILABLE):
    print("skipping MOFA-Flex-GraphGP pilot - mofaflex-graphgp fork not available")
else:
    for matrix_name, X in MATRICES.items():
        eigvecs = ppi_eigvecs_by_matrix.get(matrix_name)
        eigvals = ppi_eigvals_by_matrix.get(matrix_name)
        if eigvecs is None:
            print(f"{matrix_name}/{METHOD_NAME}: skipped - no graph eigenbasis (see Part 1)")
            continue
        matrix_path = _pilot_matrix_path(matrix_name, METHOD_NAME)
        if (matrix_name, METHOD_NAME) in _pilot_row_index and matrix_path.exists():
            print(f"{matrix_name}/{METHOD_NAME}: skipped - checkpoint + saved matrix both present")
            continue
        print(f"\n=== {METHOD_NAME}: {matrix_name} {X.shape} ===")
        Y_masked, mask, mechanism = build_masked_scenario(PILOT_ROW, X)
        t0 = time.time()
        try:
            X_hat_raw, diag = mofa_flex_graphgp_impute(Y_masked, eigvecs, eigvals, seed=RNG_SEED)
            m, X_hat = score_full(matrix_name, METHOD_NAME, X, X_hat_raw, mask, t0)
            n_epochs_run = diag["n_epochs_run"]
            max_epochs_run = diag["max_epochs"]
            stop_reason = "early-stopped" if diag["early_stopped"] else "hit max_epochs"
            print(f"  {n_epochs_run}/{max_epochs_run} epochs run ({stop_reason})")
            if diag.get("weight_prior_lengthscales") is not None:
                _ls = diag["weight_prior_lengthscales"]
                _ls_arr = next(iter(_ls.values())) if isinstance(_ls, dict) else _ls
                print(f"  graph-kernel lengthscale(s): shape={np.asarray(_ls_arr).shape} "
                      f"(shorter = graph prior shrinks loadings toward network neighbors more strongly)")
            elif "lengthscale_error" in diag:
                print(f"  (lengthscale diagnostic unavailable: {diag['lengthscale_error']})")
        except Exception as exc:
            m = {"Matrix": matrix_name, "Method": METHOD_NAME, "Seconds": time.time() - t0,
                 "RMSE": np.nan, "Error": f"{type(exc).__name__}: {exc}"}
            X_hat = None
            print(f"  !! {METHOD_NAME} failed on {matrix_name}: {exc}")
        if X_hat is not None:
            _record_pilot_result(matrix_name, METHOD_NAME, m, X_hat)
        rmse_val = m.get("RMSE", float("nan"))
        print(f"  RMSE={rmse_val:.4f}  ({m['Seconds']:.2f}s)")
        print(f"  - multiply by however many full-grid scenarios you plan to run (Cell 8) for a "
              f"rough time estimate before committing to it - GraphGP is typically slower per "
              f"epoch than plain MOFA-Flex (SpikeSlab), given the extra graph-kernel computation.")


dlbcl_rna/MOFA-Flex-GraphGP: skipped - checkpoint + saved matrix both present
dlbcl_protein/MOFA-Flex-GraphGP: skipped - checkpoint + saved matrix both present

=== MOFA-Flex-GraphGP: crc_protein (500, 997) ===


INFO: Initializing factors using 'random' method...
INFO: Guessed max_plate_nesting = 3


  2000/2000 epochs run (hit max_epochs)
  graph-kernel lengthscale(s): shape=(15,) (shorter = graph prior shrinks loadings toward network neighbors more strongly)
  RMSE=1.3447  (121.50s)
  - multiply by however many full-grid scenarios you plan to run (Cell 8) for a rough time estimate before committing to it - GraphGP is typically slower per epoch than plain MOFA-Flex (SpikeSlab), given the extra graph-kernel computation.


In [8]:
# --- Cell 8: full-grid MOFA-Flex-GraphGP, checkpointed into benchmark_checkpoint_<matrix>.csv ----
# Mirrors evaluate_mofaflex_multiomic_clinical.ipynb's own Cell 7 (run_mofaflex_full_grid /
# MOFAFLEX_FULL_GRID_ALL_REPLICATES), which itself mirrors evaluate_mofaplus_multiomic_
# clinical.ipynb's full-grid cell - same MASKING_FRACTIONS/MNAR_FRACTIONS/N_REPLICATES,
# same Seed formula (RNG_SEED * 10_000 + Scenario_ID), same SLOW-tier scaffolding (a no-op
# with the current single-value MASKING_FRACTIONS grid, kept only so this cell doesn't
# silently diverge if the grid is widened again later) - so MOFA-Flex-GraphGP's
# Scenario_ID/Seed line up exactly with MOFA+'s, MOFA-Flex's, and the 25 conventional
# methods', and all of their rows merge cleanly in evaluate_imputation_methods_clinical.ipynb.
#
# Checkpointed into the SAME benchmark_checkpoint_<matrix>.csv files every other full-grid
# cell in this project writes to. Don't run this cell at the same time as those other
# full-grid cells for the same matrix (both do a read-modify-write against the same
# checkpoint file); run them one after another instead - each resumes from whatever is
# already checkpointed.
#
# GraphGP is typically slower per epoch than plain MOFA-Flex (the graph-kernel computation
# adds real overhead on top of SpikeSlab's), on top of the STRING graph build itself
# (Part 1, a one-time cost per matrix, cached to disk). Budget real wall-clock time before
# running this on all three matrices with MOFAFLEX_GRAPHGP_FULL_GRID_ALL_REPLICATES=True
# (5x the scenario count of Replicate == 1 alone). Time one matrix first.

MASKING_FRACTIONS = [0.30]
MNAR_FRACTIONS = [0.00, 0.25, 0.50, 0.75, 1.00]
N_REPLICATES = 5  # -> 25 scenarios total, same grid as every other full-grid cell in this project

SLOW_GRID_MASKING = {0.30}  # matches benchmark_imputation_clinical.ipynb/py's SLOW_NAMES tier
SLOW_GRID_MNAR = {0.00, 0.25, 0.50, 0.75, 1.00}

# Mirrors MOFA-Flex's MOFAFLEX_FULL_GRID_ALL_REPLICATES / MOFA+_native's MOFAPLUS_FULL_GRID_
# ALL_REPLICATES: True runs MOFA-Flex-GraphGP on every replicate, not just Replicate == 1.
# Set to False to fall back to the Replicate == 1 -only subset (5 scenarios/matrix) if the
# full 25/matrix is too slow.
MOFAFLEX_GRAPHGP_FULL_GRID_ALL_REPLICATES = True

_rows = []
_scenario_id = 0
for mf in MASKING_FRACTIONS:
    for nf in MNAR_FRACTIONS:
        for rep in range(1, N_REPLICATES + 1):
            _scenario_id += 1
            _rows.append({"Scenario_ID": _scenario_id, "Masking_Fraction": mf, "MNAR_Fraction": nf,
                          "Replicate": rep, "Seed": RNG_SEED * 10_000 + _scenario_id})
experiment_design = pd.DataFrame(_rows)

_replicate_ok = (
    experiment_design["Replicate"] >= 1  # no-op: every replicate qualifies
    if MOFAFLEX_GRAPHGP_FULL_GRID_ALL_REPLICATES
    else experiment_design["Replicate"] == 1
)
mofaflex_graphgp_full_grid_design = experiment_design[
    _replicate_ok
    & (experiment_design["Masking_Fraction"].isin(SLOW_GRID_MASKING))
    & (experiment_design["MNAR_Fraction"].isin(SLOW_GRID_MNAR))
].reset_index(drop=True)
print(
    f"MOFA-Flex-GraphGP full-grid scenario subset: {len(mofaflex_graphgp_full_grid_design)} of "
    f"{len(experiment_design)} scenarios x {len(MATRICES)} matrices = "
    f"{len(mofaflex_graphgp_full_grid_design) * len(MATRICES)} total training runs"
)

FULL_GRID_METHOD_NAME = "MOFA-Flex-GraphGP"


def run_mofaflex_graphgp_full_grid(matrix_name, X, eigvecs, eigvals, checkpoint_path):
    """Same skip/resume/checkpoint-after-every-scenario contract as
    run_mofaflex_full_grid / run_mofaplus_full_grid / run_full_benchmark(), restricted to
    mofaflex_graphgp_full_grid_design and a single method."""
    if checkpoint_path.exists():
        all_results = pd.read_csv(checkpoint_path).to_dict("records")
        print(f"{matrix_name}: resuming from checkpoint, {len(all_results)} result(s) already in file.")
    else:
        all_results = []
    # Only rows WITHOUT a recorded Error count as "done" - a scenario that crashed on
    # a previous run (e.g. the fully-missing-feature NaN-init crash)
    # is retried automatically on resume instead of being silently skipped forever.
    already_done = {(r["Scenario_ID"], r["Method"]) for r in all_results if pd.isna(r.get("Error"))}
    n_failed_retry = sum(
        1 for r in all_results
        if pd.notna(r.get("Error")) and r["Method"] == FULL_GRID_METHOD_NAME
        and (r["Scenario_ID"], r["Method"]) not in already_done
    )
    if n_failed_retry:
        print(f"{matrix_name}: {n_failed_retry} previously-failed {FULL_GRID_METHOD_NAME} "
              f"scenario(s) will be retried this run.")

    for _, row in mofaflex_graphgp_full_grid_design.iterrows():
        if (row["Scenario_ID"], FULL_GRID_METHOD_NAME) in already_done:
            continue
        Y_masked, mask, mechanism = build_masked_scenario(row, X)
        mcar_only = pd.DataFrame(mechanism.values == "MCAR", index=mask.index, columns=mask.columns)
        mnar_only = pd.DataFrame(mechanism.values == "MNAR", index=mask.index, columns=mask.columns)
        t0 = time.time()
        try:
            # seed=row["Seed"], not the module-level RNG_SEED default - each scenario gets
            # its own SVI training initialization, tied to its own scenario the same way
            # build_masked_scenario above already derives its masking rng from row["Seed"].
            X_hat_raw, diag = mofa_flex_graphgp_impute(Y_masked, eigvecs, eigvals, seed=int(row["Seed"]))
            X_hat = X_hat_raw.reindex(index=X.index, columns=X.columns)
            overall = calculate_metrics(X, X_hat, mask)
            rec = {"Scenario_ID": row["Scenario_ID"], "Method": FULL_GRID_METHOD_NAME,
                   "Masking_Fraction": row["Masking_Fraction"], "MNAR_Fraction": row["MNAR_Fraction"],
                   "Replicate": row["Replicate"], "Seconds": time.time() - t0, **overall}
            if mcar_only.values.any():
                rec["RMSE_MCAR_subset"] = calculate_metrics(X, X_hat, mcar_only)["RMSE"]
            if mnar_only.values.any():
                rec["RMSE_MNAR_subset"] = calculate_metrics(X, X_hat, mnar_only)["RMSE"]
        except Exception as exc:
            rec = {"Scenario_ID": row["Scenario_ID"], "Method": FULL_GRID_METHOD_NAME,
                   "Masking_Fraction": row["Masking_Fraction"], "MNAR_Fraction": row["MNAR_Fraction"],
                   "Replicate": row["Replicate"], "Seconds": time.time() - t0, "RMSE": np.nan,
                   "Error": f"{type(exc).__name__}: {exc}"}
            print(f"    !! {matrix_name}/{FULL_GRID_METHOD_NAME} failed on scenario {row['Scenario_ID']}: {exc}")
        all_results.append(rec)
        already_done.add((row["Scenario_ID"], FULL_GRID_METHOD_NAME))
        pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)  # checkpoint after each scenario
        rmse_disp = rec.get("RMSE", float("nan"))
        print(
            f"  scenario {int(row['Scenario_ID']):3d} (mask={row['Masking_Fraction']:.2f}, "
            f"mnar={row['MNAR_Fraction']:.2f}): RMSE={rmse_disp:.4f} ({rec['Seconds']:.1f}s)"
        )

    return pd.DataFrame(all_results)


# Run one matrix at a time (comment out if you don't want to run full-grid) - each call resumes from its own checkpoint
# if interrupted, so it's safe to run matrices in separate sessions. Time the first matrix
# before committing to all three.
mofaflex_graphgp_full_grid_results = {}
if MOFAFLEX_AVAILABLE and GRAPHGP_AVAILABLE:
     for matrix_name, X in MATRICES.items():
         eigvecs = ppi_eigvecs_by_matrix.get(matrix_name)
         eigvals = ppi_eigvals_by_matrix.get(matrix_name)
         if eigvecs is None:
             print(f"=== full-grid MOFA-Flex-GraphGP: {matrix_name} skipped - no graph eigenbasis (see Part 1) ===")
             continue
         print(f"\n=== full-grid MOFA-Flex-GraphGP: {matrix_name} {X.shape} ===")
         mofaflex_graphgp_full_grid_results[matrix_name] = run_mofaflex_graphgp_full_grid(
             matrix_name, X, eigvecs, eigvals, RESULTS_DIR / f"benchmark_checkpoint_{matrix_name}.csv"
         )


MOFA-Flex-GraphGP full-grid scenario subset: 25 of 25 scenarios x 3 matrices = 75 total training runs

=== full-grid MOFA-Flex-GraphGP: dlbcl_rna (500, 318) ===
dlbcl_rna: resuming from checkpoint, 475 result(s) already in file.

=== full-grid MOFA-Flex-GraphGP: dlbcl_protein (500, 332) ===
dlbcl_protein: resuming from checkpoint, 739 result(s) already in file.

=== full-grid MOFA-Flex-GraphGP: crc_protein (500, 997) ===
crc_protein: resuming from checkpoint, 700 result(s) already in file.
